In [ ]:
!git clone https://github.com/LiveCodeBench/LiveCodeBench.git
%cd /content/LiveCodeBench
!pip install uv
!uv pip install --system -e .

# Force downgrade to a version of Anthropic that still contains HUMAN_PROMPT
!uv pip install --system "anthropic<1.0.0"

: 

In [ ]:
!uv pip install --system "datasets<4"
from lcb_runner.benchmarks.code_generation import load_code_generation_dataset
from lcb_runner.prompts.code_generation import format_prompt_generation
from lcb_runner.lm_styles import LMStyle
from lcb_runner.evaluation.compute_code_generation_metrics import codegen_metrics

In [ ]:
!apt-get update -qq && apt-get install -y -qq iputils-ping
!ping -c 4 openai.rc.asu.edu
import json
serverinfo_ = !curl ipinfo.io
info_ = json.loads(''.join(serverinfo_))
print(json.dumps(info_,indent=2))
if info_["country"] != "US":
  print(f"Currently connected to {info_['country']} (region:{info_['region']})")
  print(f"Please connect to a US server!")
  print(f"Disconnecting")
  from google.colab import runtime
  runtime.unassign()
  raise Exception("ServerLocaleError")

In [ ]:
from google.colab import userdata

# Safely fetch the user's hidden secret
api_token = userdata.get('OPENAI_KEY')

In [ ]:
from openai import OpenAI

url = "https://openai.rc.asu.edu/v1"
client = OpenAI(api_key=api_token,
                base_url=url
                )

In [ ]:
# Search for problems
import pandas as pd
problems = load_code_generation_dataset(release_version="release_v5")  # keep this fixed

df = pd.DataFrame([{
    "id": x.question_id, "title": x.question_title,
    "platform": x.platform.value, "difficulty": x.difficulty.value,
    "date": x.contest_date.date()} for x in problems])

# Filter list of problems
display(df[(df.platform == "leetcode") & (df.difficulty == "easy")].sort_values("date").head(40))

# from datasets import load_dataset
# from lcb_runner.benchmarks.code_generation import CodeGenerationProblem
# import pandas as pd

# ds = load_dataset("livecodebench/code_generation_lite", split="test",
#                   version_tag="release_v5", trust_remote_code=True)
# print(ds.column_names)   # confirm the column names below exist

# df = ds.select_columns(["question_id", "question_title", "platform",
#                         "difficulty", "contest_date"]).to_pandas()

# # filter list of problems
# display(df[(df.platform == "leetcode") & (df.difficulty == "easy")].sort_values("contest_date").head(40))

In [ ]:
PROBLEM_ID = "2857"
p = next((x for x in problems if x.question_id == PROBLEM_ID), None)
assert p is not None, f"No problem with id {PROBLEM_ID}"
print(p.question_title, "|", p.difficulty.value, "|", p.contest_date.date())
print(p.question_content)
print(p.starter_code)

In [ ]:
import re

def call_llm(messages, temperature=0.2):
    r = client.chat.completions.create(
        model="gpt-oss-120b", messages=messages, temperature=temperature)
    return r.choices[0].message.content

def extract_code(text):
    blocks = re.findall(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return blocks[-1] if blocks else text   # same rule for both prompts

MY_SYSTEM = ("Read through the following specification and respond with error-free Python code "
             "that will solve the problem with maximum space and time efficiency. "
             "Stay within the constraints that are provided. Ensure whitespace is utilized and use comments minimally. "
             "Respond with only the Python code and nothing else. Make no mistakes, and ensure that the code has no errors. "
             "Each specification will also have starter code, if that is not provided, do NOT write a solution.")

my_messages = [
    {"role": "system", "content": MY_SYSTEM},
    {"role": "user", "content": p.question_content + "\n\nStarter code:\n" + p.starter_code},
]

lcb_prompt = format_prompt_generation(p, LMStyle.OpenAIChat)
print(type(lcb_prompt)); print(lcb_prompt)   # inspect this once
lcb_messages = lcb_prompt if isinstance(lcb_prompt, list) else [{"role": "user", "content": lcb_prompt}]


In [ ]:
my_code  = call_llm(my_messages)
lcb_code = call_llm(lcb_messages)

In [ ]:
with open("output.py", "w", encoding="utf-8") as f:
    f.write(my_code)

In [ ]:
lcb_code = extract_code(lcb_code)
with open("output_lcb.py", "w", encoding="utf-8") as f:
    f.write(lcb_code)

In [ ]:
sample = p.get_evaluation_sample()
for name, code in [("my prompt", my_code), ("LCB prompt", lcb_code)]:
    metrics, results, _ = codegen_metrics([sample], [[code]], k_list=[1],
                                          num_process_evaluate=1, timeout=6)
    print(name, metrics["pass@1"])